<a href="https://colab.research.google.com/github/BreakTheAlgorithm/MLforALL/blob/main/book_ch11_clustering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div style='background:linear-gradient(135deg,#1B3A6B,#2D5AA0);padding:30px;border-radius:12px;color:white;font-family:Arial'>
<h1 style='margin:0;font-size:28px'>🔵 Chapter 11 — Unsupervised Learning: Clustering</h1>
<p style='margin:8px 0 0'>Book: Machine Learning — From Zero to Engineer | Est. Time: 70 mins | Level: Intermediate</p>
</div>

## 📋 Learning Objectives

By the end of this notebook, you will be able to:
- Explain the difference between supervised and unsupervised learning
- Apply K-Means and DBSCAN clustering algorithms
- Use the Elbow Method and Silhouette Score to choose the optimal k
- Visualise clusters with scatter plots and cluster profiles
- Apply clustering to a real customer segmentation problem

---

In [ ]:
# ─────────────────────────────────────────────────────────────
# Setup & Imports
# ─────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.cluster     import KMeans, DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.metrics      import silhouette_score
from sklearn.datasets     import make_blobs, make_moons

%matplotlib inline
np.random.seed(42)

# ── Customer dataset: Indian e-commerce customers ─────────────
n = 500
customers = pd.DataFrame({
    'annual_spend':   np.random.lognormal(11, 0.8, n).astype(int),
    'num_orders':     np.random.randint(1, 100, n),
    'avg_order_value': np.random.randint(500, 10000, n),
    'days_since_last': np.random.randint(1, 365, n),
    'age':            np.random.randint(18, 65, n),
})
print('Customer dataset shape:', customers.shape)
customers.describe()

## 📖 Section A — K-Means Clustering

K-Means partitions data into k clusters by minimising within-cluster variance (inertia):

1. Randomly initialise k centroids
2. Assign each point to nearest centroid
3. Update centroids to mean of assigned points
4. Repeat until convergence

```python
from sklearn.cluster import KMeans
km = KMeans(n_clusters=3, random_state=42, n_init=10)
km.fit(X)
labels    = km.labels_         # cluster assignment per sample
centroids = km.cluster_centers_# k centroid coordinates
inertia   = km.inertia_        # within-cluster sum of squares
```

> 💡 **Key Rule:** Always scale features before K-Means. A feature with large values (salary ₹10M) will dominate the distance calculation over a feature with small values (age 25–65). StandardScaler puts them on the same scale.

In [ ]:
# ─────────────────────────────────────────────────────────────
# Demo: K-Means on synthetic data
# ─────────────────────────────────────────────────────────────
X_blobs, _ = make_blobs(n_samples=300, centers=4, cluster_std=1.0, random_state=42)

km = KMeans(n_clusters=4, random_state=42, n_init=10)
km.fit(X_blobs)

plt.figure(figsize=(7, 5))
scatter = plt.scatter(X_blobs[:, 0], X_blobs[:, 1],
                      c=km.labels_, cmap='Set1', alpha=0.7, s=30)
plt.scatter(km.cluster_centers_[:, 0], km.cluster_centers_[:, 1],
            c='black', s=200, marker='X', zorder=5, label='Centroids')
plt.title('K-Means Clustering (k=4)')
plt.legend()
plt.show()
print(f'Inertia: {km.inertia_:.1f}  |  Silhouette: {silhouette_score(X_blobs, km.labels_):.4f}')

## 📖 Section B — Choosing k: Elbow Method and Silhouette Score

```python
inertias, silhouettes = [], []
for k in range(2, 11):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X, km.labels_))
# Elbow: look for 'elbow' in inertia curve
# Silhouette: higher is better (range -1 to 1)
```

> 💡 **Key Rule:** The elbow point is where adding more clusters gives diminishing returns in inertia reduction. Silhouette score measures how similar a point is to its own cluster vs other clusters — 1.0 is perfect, 0.0 is ambiguous, negative means misclassified.

---
## 🟢 Exercise 11.1 — Elbow Method and Silhouette *(Level 1 · Est. 10 min)*

> Scale the customer dataset. Plot inertia and silhouette score for k=2 to k=10. Identify the best k.

In [ ]:
# ─────────────────────────────────────────────────────────────
# Exercise 11.1: Elbow Method + Silhouette
# ─────────────────────────────────────────────────────────────
scaler = StandardScaler()
X_cust = scaler.fit_transform(customers)

inertias, silhouettes = [], []
k_range = range(2, 11)

# YOUR CODE HERE — compute inertia and silhouette for each k

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
# Plot elbow
# YOUR CODE HERE
# Plot silhouette
# YOUR CODE HERE

best_k = # YOUR CODE HERE
print(f'Best k by silhouette: {best_k}')

assert len(inertias) == len(k_range)
assert len(silhouettes) == len(k_range)
print('✅ Elbow and silhouette analysis complete!')

In [ ]:
# @title ✅ Solution — Exercise 11.1 (click to expand)
scaler = StandardScaler()
X_cust = scaler.fit_transform(customers)

inertias, silhouettes = [], []
k_range = range(2, 11)
for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_cust)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_cust, km.labels_))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(list(k_range), inertias, 'bo-')
axes[0].set_title('Elbow Method — Inertia')
axes[0].set_xlabel('k')
axes[0].set_ylabel('Inertia (WCSS)')

axes[1].plot(list(k_range), silhouettes, 'ro-')
axes[1].set_title('Silhouette Score')
axes[1].set_xlabel('k')
axes[1].set_ylabel('Silhouette')

plt.tight_layout()
plt.show()

best_k = list(k_range)[np.argmax(silhouettes)]
print(f'Best k by silhouette: {best_k}  (score={max(silhouettes):.4f})')
print('✅ Use silhouette to pick k quantitatively — the elbow is subjective.')

---
## 🟡 Exercise 11.2 — Customer Segmentation *(Level 2 · Est. 20 min)*

> Cluster customers into the best k segments. Describe each cluster with mean values. Name each segment (e.g., 'High Value', 'At Risk', etc.).

In [ ]:
# ─────────────────────────────────────────────────────────────
# Exercise 11.2: Customer Segmentation
# ─────────────────────────────────────────────────────────────

# Fit K-Means with best_k
# YOUR CODE HERE

# Add cluster labels to customers DataFrame
# YOUR CODE HERE

# Profile each cluster with mean values
cluster_profiles = # YOUR CODE HERE
print('Cluster Profiles:')
print(cluster_profiles.round(0))

# Visualise clusters (annual_spend vs num_orders, colour = cluster)
# YOUR CODE HERE

assert 'cluster' in customers.columns
print('\n✅ Customer segmentation complete!')

In [ ]:
# @title ✅ Solution — Exercise 11.2 (click to expand)

km_best = KMeans(n_clusters=best_k, random_state=42, n_init=10)
customers['cluster'] = km_best.fit_predict(X_cust)

cluster_profiles = customers.groupby('cluster').mean().round(0)
print('Cluster Profiles (means):')
print(cluster_profiles)

plt.figure(figsize=(8, 5))
sns.scatterplot(data=customers, x='annual_spend', y='num_orders',
                hue='cluster', palette='Set1', alpha=0.7, s=40)
plt.title('Customer Segments: Annual Spend vs Number of Orders')
plt.xlabel('Annual Spend (₹)')
plt.ylabel('Number of Orders')
plt.show()

print('\nSegment naming guide:')
print('- High spend + many orders = Champion customers')
print('- High spend + few orders  = Big ticket buyers')
print('- Low spend + low recency  = At-risk / lapsed')
print('- Low spend + recent       = New customers')

---
## 🔴 Exercise 11.3 — DBSCAN vs K-Means on Non-Globular Data *(Level 3 · Est. 20 min)*

> Generate moon-shaped data. Fit K-Means(k=2) and DBSCAN. Visualise and explain why K-Means fails here.

In [ ]:
# @title ✅ Solution — Exercise 11.3 (click to expand)

X_moons, y_moons = make_moons(n_samples=300, noise=0.1, random_state=42)

km_moons = KMeans(n_clusters=2, random_state=42, n_init=10)
km_labels = km_moons.fit_predict(X_moons)

db = DBSCAN(eps=0.3, min_samples=5)
db_labels = db.fit_predict(X_moons)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].scatter(X_moons[:, 0], X_moons[:, 1], c=y_moons, cmap='Set1', s=20, alpha=0.8)
axes[0].set_title('True Clusters (moons)')

axes[1].scatter(X_moons[:, 0], X_moons[:, 1], c=km_labels, cmap='Set1', s=20, alpha=0.8)
axes[1].set_title('K-Means (k=2) — FAILS')

axes[2].scatter(X_moons[:, 0], X_moons[:, 1], c=db_labels, cmap='Set1', s=20, alpha=0.8)
axes[2].set_title('DBSCAN — Correctly Identifies Moons')

plt.suptitle('K-Means assumes spherical clusters — fails on arbitrary shapes', fontsize=12)
plt.tight_layout()
plt.show()
print('\nK-Means limitation: it assumes clusters are convex and similarly sized.')
print('DBSCAN finds clusters of arbitrary shape based on density — no need to specify k.')

---
## 🎤 Interview Q&A

<details>
<summary><strong>Q1: What are the limitations of K-Means?</strong></summary>

**Answer:** (1) You must specify k in advance. (2) Assumes spherical, similarly-sized clusters — fails on elongated or complex shapes. (3) Sensitive to outliers — outliers distort centroids. (4) Non-deterministic — results depend on random initialisation (use n_init > 1 or kmeans++ initialisation). (5) Only works well with numeric data — categorical requires special treatment.
</details>

<details>
<summary><strong>Q2: How does DBSCAN work?</strong></summary>

**Answer:** DBSCAN (Density-Based Spatial Clustering) defines clusters as dense regions separated by sparse regions. Two parameters: eps (neighbourhood radius) and min_samples (minimum points to form a dense region). A core point has ≥ min_samples neighbours within eps. Cluster = all points density-reachable from a core point. Points not reachable from any core point are noise (-1 label). Advantage: finds arbitrary shapes, automatically detects outliers.
</details>

<details>
<summary><strong>Q3: What is the Silhouette Score?</strong></summary>

**Answer:** Silhouette score for a point = (b − a) / max(a, b), where a = mean distance to same-cluster points (cohesion), b = mean distance to nearest other-cluster points (separation). Range: -1 to +1. Closer to +1 → well clustered. Close to 0 → on boundary. Negative → may be misassigned. Average silhouette score across all points evaluates overall clustering quality — use it to compare different k values.
</details>

---

<div style='background:#EDF7F0;border-left:6px solid #2E7D50;padding:20px;border-radius:8px;margin-top:20px'>
<h3 style='color:#2E7D50'>✅ Chapter 11 Complete!</h3>
<ul>
<li>K-Means: algorithm, implementation, elbow method</li>
<li>Silhouette score for k selection</li>
<li>Customer segmentation on real-world style data</li>
<li>DBSCAN for non-globular cluster shapes</li>
</ul>
<p><strong>Next:</strong> Chapter 12 — Dimensionality Reduction</p>
</div>